# MCTS Tutorial: Tic Tac Toe Testbed

This notebook is the canonical Tic Tac Toe walkthrough for the project. The goal is to understand the implemented MCTS data flow on a tiny solved game before using the same algorithm on Connect Four.

Tic Tac Toe is useful because the whole state is visible, legal actions are easy to inspect, and exact minimax values are available in the GUI. Connect Four uses the same MCTS interface but has a larger branching factor, deeper tactics, and experiment settings such as rollout policy and exploration constant sweeps.

## Commands To Run

From the project root, after `uv sync --dev` and `source .venv/bin/activate`:

```bash
# Text trace. --verbose is required to print per-move search details.
tictactoe-mcts --iterations 300 --seed 0 --verbose

# Interactive visualizer with exact minimax move values.
tictactoe-play

# Self-play sweep used to show convergence toward tied perfect play.
tictactoe-mcts-sweep --iterations 1,5,10,50,100,500,1000 --games 100 --seed 0 --progress-every 10
```

The script under `src/connect4/tictactoe/tictactoe_mcts.py` is intentionally just runnable code. The explanation lives here.

## Scope Of This Testbed

What this notebook covers:

- the game-independent MCTS state interface,
- Tic Tac Toe state transitions,
- what an MCTS node stores,
- UCT selection, expansion, random rollout, and backpropagation,
- how to interpret root action visits and values,
- how the pygame visualizer helps debug decisions.

What this notebook does **not** cover:

- Connect Four column actions,
- large branching/depth behavior,
- exploration-constant sweeps,
- heuristic-guided rollouts,
- offline value-network cutoffs,
- final report experiment pipelines.

Those are Connect Four experiments, not Tic Tac Toe tutorial features.

## 1. Imports And State Interface

MCTS only requires a deterministic two-player state with this interface:

```python
state.current_player
state.is_terminal
state.legal_actions()
state.next_state(action)
state.result_for(player)
```

Tic Tac Toe and Connect Four both implement this interface, which is why `connect4.mcts.MCTS` does not need game-specific logic.

In [ ]:
from connect4.mcts import MCTS, MCTSNode
from connect4.tictactoe.tictactoe_mcts import PLAYER_O, PLAYER_X, TicTacToeState

state = TicTacToeState()
print(state.render_ascii())
print('current_player =', state.current_player)
print('legal_actions =', state.legal_actions())

## 2. State Representation And Actions

The tutorial uses integer board values:

```text
X = 1
O = -1
empty = 0
```

Actions are flattened cell indexes:

```text
0 1 2
3 4 5
6 7 8
```

So action `4` means center square. The state returns copied future states so tree search can explore hypothetical moves without mutating the real board.

In [ ]:
after_center = state.next_state(4)
print('Original board:')
print(state.render_ascii())
print('
After X plays center:')
print(after_center.render_ascii())
print('next current_player =', after_center.current_player)

## 3. What An MCTS Node Stores

Each node stores one state plus search statistics:

- `parent`: previous node in the tree,
- `action`: move taken at the parent to reach this node,
- `untried_actions`: legal moves not expanded yet,
- `children`: expanded child nodes keyed by action,
- `visits`: number of simulations through the node,
- `value_sum`: total rollout value from the root player's perspective.

The mean value shown in outputs is `value_sum / visits`.

In [ ]:
root = MCTSNode(state=state)
print('untried_actions =', root.untried_actions)
print('children =', root.children)
print('visits =', root.visits)
print('value_sum =', root.value_sum)
print('mean_value =', root.mean_value)

## 4. One Search From The Empty Board

`MCTS.search(state)` runs repeated iterations:

```text
selection -> expansion -> rollout -> backpropagation
```

The result chooses the root child with the most visits. The `action_stats` table reports each root action's visits and mean rollout value.

In [ ]:
result = MCTS(iterations=50, seed=0).search(state)
print('chosen action =', result.action)
print('root visits =', result.root_visits)
for action, stats in result.action_stats.items():
    print(action, stats)

## 5. UCT Selection

Selection uses a UCB1-style rule at each tree node:

```text
score = mean_value + c * sqrt(log(parent_visits) / child_visits)
```

`mean_value` is exploitation: moves that have looked good so far.

The square-root term is exploration: moves that have not been sampled much.

In adversarial games, opponent nodes flip the exploitation sign. At root-player nodes, high root value is good. At opponent nodes, low root value is good for the opponent.

## 6. Root Player Perspective

Every search stores values from the player who started that search:

```python
root_player = int(root_state.current_player)
terminal_value = terminal_state.result_for(root_player)
```

Value convention:

```text
 1.0 = root player wins
 0.0 = draw
-1.0 = root player loses
```

This means a positive MCTS value is always good for the player who is deciding the current move, whether that player is X or O.

In [ ]:
next_state = state.next_state(result.action)
print(next_state.render_ascii())
print('next player =', next_state.current_player)

## 7. Full Self-Play Trace In Python

This mirrors the `tictactoe-mcts --verbose` command. Each move starts a fresh search from the current state.

In [ ]:
game = TicTacToeState()
move_number = 1
while not game.is_terminal:
    result = MCTS(iterations=100, seed=move_number).search(game)
    player = 'X' if game.current_player == PLAYER_X else 'O'
    print(f'
Move {move_number}: {player} chooses {result.action}')
    for action, stats in result.action_stats.items():
        print(f"  {action}: visits={int(stats['visits'])}, value={stats['mean_value']:.2f}")
    game = game.next_state(result.action)
    print(game.render_ascii())
    move_number += 1

winner = 'X' if game.winner == PLAYER_X else 'O' if game.winner == PLAYER_O else 'Tie'
print('
Result:', winner)

## 8. GUI Visualizer

Run:

```bash
tictactoe-play
```

The GUI can switch between human-human, human-AI, and AI-AI. It also shows exact minimax move values for the current player:

```text
+1 = current player can force a win after that move
 0 = perfect play should draw
-1 = current player loses against perfect response
```

This exact overlay is specific to Tic Tac Toe because the game is small enough to solve directly. Connect Four uses approximate MCTS evaluation instead.

## 9. Transfer To Connect Four

Connect Four exposes the same MCTS interface:

```python
from connect4.core import ConnectFourState
from connect4.mcts import MCTS

state = ConnectFourState.new()
result = MCTS(iterations=800, seed=0).search(state)
state = state.next_state(result.action)
```

The action meaning changes: Tic Tac Toe actions are cells, Connect Four actions are columns. MCTS does not care because game-specific rules live in the state object.

## 10. Where To Find Connect Four Commands

Connect Four files and report-style commands:

```bash
connect4-play
connect4-evaluate c-sweep --iterations 100,500 --exploration-weights 0.25,0.5,1.0,sqrt2,2.0,4.0 --games-per-side 20 --seed 0
connect4-evaluate rollout-policy-sweep --policies random,heuristic --iterations 100,500 --games-per-side 20 --seed 0
```

Main files: `src/connect4/core.py`, `src/connect4/mcts.py`, `src/connect4/agents.py`, and `src/connect4/scripts/evaluate_agents.py`. Use small game counts for smoke tests and larger game counts for report figures.